# 파이썬 사전 문법 익혀보기
- 그래프 형태로 만들 때 주의할 점 : 타입이 맞아야 한다
- 더 정확히 말하자면 LangGraph에서는 State의 키와 각 노드가 반환하는 값의 구조가 State 스키마와 맞아야 한다
- 딕셔너리 : 예) name : "홍길동", prompt : "메타버스가 뭐에요?", "오늘 주식은 오르는가요?"

- 즉, LangGraph에서 State는 여러 노드가 공유하는 딕셔너리이다.
- TypedDict로 State 구조를 정의하면 각 노드가 어떤 키를 읽고, 어떤 키를 반환해야 하는지 명확해진다.

## 1.TypedDict
- 딕셔너리(dict) 에 키와 값의 타입(형식) 을 미리 정의해두는 문법
- dict 와의 차이 : 코드 작성할 때 타입 체커가 타입을 체크해서 오류를 미리 잡아내는 것
- dict는 추후에 추가적인 키 입력할 수 있지만 TypedDict 미리 정의된 구조를 따르도록 해야 에러가 덜 발생
- 코드가 어떤 형태인지 -> 가독성
- 즉, 딕셔너리인데 타입 힌트를 엄격하게 적용한 버전

[참고] vs code에서 타입 검사 기능 활성화
1. Pylance 설치
2. `Ctrl + ,`
3. 검색창에 `type checking mode` 검색
4. `Python › Analysis: Type Checking Mode` 에서 `basic`으로 변경

In [1]:
from typing import Dict, TypedDict

In [2]:
def add_price(a, b):
    return a + b

In [ ]:
def add_price(a:int, b:int) -> int:
    return a + b

print(add_price('1000', 200)) # 오류(TypeError) 발생

In [4]:
tmp_dic : Dict[str, str] = {
    'name' : '김양봉',
    'prompt' : '오늘 주식 올라요? 사요?',
    'timestamp' : '20260625'
}

In [5]:
tmp_dic['prompt'] = '오늘 날씨 어때요?'
tmp_dic['timestamp'] = 20260101
tmp_dic

{'name': '김양봉', 'prompt': '오늘 날씨 어때요?', 'timestamp': 20260101}

In [6]:
# 딕셔너리의 구조를 미리 정해둠
# stock은 일반 클래스처럼 보이지만, 실제 목적은 객체를 만드는 것이 아니라 딕셔너리의 설계도를 만드느 것.
class stock(TypedDict):
    name : str
    prompt : str
    timestamp : int

In [7]:
typed_dic : stock = {
    "name" : "장대양봉",
    "prompt" : "삼전 급등? 지금 사요?",
    "timestamp" : 230323
}

In [8]:
typed_dic

{'name': '장대양봉', 'prompt': '삼전 급등? 지금 사요?', 'timestamp': 230323}

In [9]:
typed_dic['season'] = '봄'

In [10]:
typed_dic["timestamp"] = "20230202"

## 2.Annotated
- 변수에 추가적인 의미나 메타데이터를 붙일 때 사용
- 기본 타입 정보 외에 "이 값에 이런 처리를 해라"는 부가 정보를 담는 용도
- 유효성 검사

In [ ]:
from typing import Annotated, List

# Annotated[기본타입, 추가정보]
name : Annotated[str, "본인의 이름"]
timestamp : Annotated[str, "입력한 시간"]

In [12]:
# 데이터 모델(BaseModel) 직접 만들어 보고, 유효성 검사가 잘 작동하는지 실험해 본다는 뜻
from pydantic import Field, BaseModel

# 각각 자료형과 제약조건(Field)를 함께 가진다.
class sk30(BaseModel):
    name : Annotated[str, Field(..., min_length=2, max_length=10, description="이름")]  # ...은 name 필드가 필수값이라는 표시
    timestamp : Annotated[str, Field(..., description="입력한 시간")]
    chat_history : Annotated[List[str], Field(min_length=1, max_length=10, description="채팅 내용")]

In [13]:
student = sk30(
    name="정주애",
    timestamp="20260625",
    chat_history=["안녕하세요 AI야 밥은 먹고 다니냐?", "네, 주애씨, 밥은 드셨나요?"]
)

In [14]:
student.chat_history.append("나는 배가 항상 고파")
student

sk30(name='정주애', timestamp='20260625', chat_history=['안녕하세요 AI야 밥은 먹고 다니냐?', '네, 주애씨, 밥은 드셨나요?', '나는 배가 항상 고파'])

In [ ]:
# uv add -U langgraph

## [실습]
- BaseModel로 StudentProfile 만들기
- 조건:
    - 이름은 2~10자
    - 나이는 1 이상
    - 관심 주제 리스트는 최소 1개
- 정상 데이터 1개, 에러 나는 데이터 2개 작성
- 에러 메시지를 읽고 어떤 조건 때문에 실패했는지 적기

In [15]:
class StudentProfile(BaseModel):
    name: Annotated[str, Field(..., min_length=2, max_length=10, description="학생 이름")]
    age: Annotated[int, Field(..., ge=1, description="나이")]
    interests: Annotated[List[str], Field(..., min_length=1, description="관심 주제 리스트")]

In [16]:
# 1. 정상 데이터
normal_student = {
    "name": "박지유",
    "age": 20,
    "interests": ["LangGraph", "AI Agent"]
}

student = StudentProfile(**normal_student)      # 딕셔너리 데이터를 StudentProfile 모델에 넣어서 검증하고 객체로 만드는 코드
print("정상 데이터 생성 성공")
print(student)


정상 데이터 생성 성공
name='박지유' age=20 interests=['LangGraph', 'AI Agent']


In [17]:
# 2. 에러 데이터 1: 
error_student_1 = {
    "name": "박",
    "age": 20,
    "interests": ["Python"]
}

student = StudentProfile(**error_student_1)

ValidationError: 1 validation error for StudentProfile
name
  String should have at least 2 characters [type=string_too_short, input_value='박', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/string_too_short

In [18]:
# 3. 에러 데이터 2: 
error_student_2 = {
    "name": "강성준",
    "age": 0,
    "interests": []
}

In [19]:
student = StudentProfile(**error_student_2)

ValidationError: 2 validation errors for StudentProfile
age
  Input should be greater than or equal to 1 [type=greater_than_equal, input_value=0, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/greater_than_equal
interests
  List should have at least 1 item after validation, not 0 [type=too_short, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.12/v/too_short